# Disagreement-Aware Probabilistic U-Net on LIDC

This is the self-contained notebook submission for the project. All project code needed to download data, define the model, train the ablations, evaluate results, and create figures is inside this notebook.

Default mode is a short smoke run so `Runtime -> Run all` completes in Colab. For final project results, change `RUN_MODE = "final"` in the configuration cell.


## 1. Install Dependencies


In [ ]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy<2", "matplotlib>=3.9", "Pillow>=10", "torch>=2.0", "tqdm>=4.66"
], check=True)


## 2. Configuration


In [ ]:
from pathlib import Path
import argparse
import csv
import json
import math
import os
import random
import shutil
import tarfile
import tempfile
import time
import urllib.request
from types import SimpleNamespace

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch import distributions
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm
from IPython.display import Image as DisplayImage, display

RUN_MODE = "smoke"  # "smoke", "pilot", or "final"
FORCE_RETRAIN = False
EVAL_SPLIT = "test"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 1

MODE_CONFIGS = {
    "smoke": {
        "steps": 2,
        "max_train": 8,
        "max_val": 4,
        "batch_size": 1,
        "eval_batch_size": 1,
        "feature_maps": 2,
        "latent_size": 2,
        "depth": 2,
        "train_samples": 2,
        "eval_samples": 2,
        "eval_every": 1,
        "save_every": 1,
        "figure_cases": 2,
        "figure_samples": 2,
    },
    "pilot": {
        "steps": 10000,
        "max_train": None,
        "max_val": 256,
        "batch_size": 16,
        "eval_batch_size": 8,
        "feature_maps": 16,
        "latent_size": 6,
        "depth": 4,
        "train_samples": 4,
        "eval_samples": 16,
        "eval_every": 1000,
        "save_every": 1000,
        "figure_cases": 8,
        "figure_samples": 4,
    },
    "final": {
        "steps": 240000,
        "max_train": None,
        "max_val": None,
        "batch_size": 32,
        "eval_batch_size": 8,
        "feature_maps": 32,
        "latent_size": 6,
        "depth": 5,
        "train_samples": 4,
        "eval_samples": 32,
        "eval_every": 1000,
        "save_every": 1000,
        "figure_cases": 8,
        "figure_samples": 4,
    },
}
CFG = MODE_CONFIGS[RUN_MODE]

OUTPUT_ROOT = Path("outputs")
DATA_ROOT = Path("data/lidc")
OUTPUT_ROOT.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("RUN_MODE:", RUN_MODE)
print("DEVICE:", DEVICE)
print(CFG)


## 3. Download And Validate LIDC Crops


In [ ]:
BASE_URL = "https://storage.googleapis.com/hpunet-data/lidc_crops"
SPLITS = ("train", "val", "test")
EXPECTED = {
    "train": (8843, 530),
    "val": (1993, 111),
    "test": (1980, 103),
}


def download_file(url, destination):
    with urllib.request.urlopen(url, timeout=60) as response:
        total = int(response.headers.get("Content-Length", 0))
        downloaded = 0
        with open(destination, "wb") as handle:
            while True:
                chunk = response.read(1 << 20)
                if not chunk:
                    break
                handle.write(chunk)
                downloaded += len(chunk)
                if total:
                    print("\r  {:.1f}/{:.1f} MB ({:.0f}%)".format(
                        downloaded / 1e6, total / 1e6, 100 * downloaded / total
                    ), end="")
        print()
    if total and downloaded != total:
        raise IOError("Truncated download: got {} of {} bytes".format(downloaded, total))


def extract_split(archive_path, split_dir):
    with tempfile.TemporaryDirectory(dir=split_dir.parent) as staging:
        with tarfile.open(archive_path, "r:gz") as tar:
            tar.extractall(staging)
        root = Path(staging)
        while True:
            entries = [p.name for p in root.iterdir()]
            if "images" in entries and "gt" in entries:
                break
            subdirs = [p for p in root.iterdir() if p.is_dir()]
            if len(subdirs) != 1:
                raise IOError("Could not locate images/ and gt/ inside {}".format(archive_path))
            root = subdirs[0]
        if split_dir.exists():
            shutil.rmtree(split_dir)
        shutil.move(str(root), str(split_dir))


def count_split(split_dir):
    images_dir = split_dir / "images"
    patients = [p for p in images_dir.iterdir() if p.is_dir()]
    images = sum(len(list(patient.glob("*.png"))) for patient in patients)
    return images, len(patients)


def download_lidc(dest=DATA_ROOT, force=False):
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    ok = True
    for split in SPLITS:
        split_dir = dest / split
        archive_path = dest / f"{split}.tar.gz"
        if split_dir.exists() and not force:
            print(f"{split}: already extracted")
        else:
            if not archive_path.exists() or force:
                print(f"{split}: downloading")
                download_file(f"{BASE_URL}/{split}.tar.gz", archive_path)
            print(f"{split}: extracting")
            extract_split(archive_path, split_dir)
            archive_path.unlink(missing_ok=True)
        images, patients = count_split(split_dir)
        expected_images, expected_patients = EXPECTED[split]
        status = "OK" if (images, patients) == (expected_images, expected_patients) else "MISMATCH"
        ok = ok and status == "OK"
        print(f"{split}: {images} images, {patients} patients (expected {expected_images} / {expected_patients}) [{status}]")
    if not ok:
        raise RuntimeError("LIDC counts differ from the published release; download may be incomplete.")
    print("Data ready under", dest.resolve())


download_lidc()


## 4. Dataset, Disagreement Targets, And Metrics


In [ ]:
NUM_GRADERS = 4


def _load_png(path):
    with Image.open(path) as handle:
        return np.asarray(handle.convert("L"))


class LIDCCrops(Dataset):
    def __init__(self, root=DATA_ROOT, split="train", crop_size=128, train=None, single_random_grader=False):
        if split not in SPLITS:
            raise ValueError(f"split must be one of {SPLITS}, got {split!r}")
        self.root = Path(root)
        self.split = split
        self.crop_size = crop_size
        self.train = (split == "train") if train is None else train
        self.single_random_grader = single_random_grader
        self.images_dir = self.root / split / "images"
        self.gt_dir = self.root / split / "gt"
        if not self.images_dir.is_dir():
            raise FileNotFoundError(f"No LIDC data at {self.root / split}; run download_lidc()")
        self.samples = self._build_index()
        if not self.samples:
            raise RuntimeError(f"Found no usable samples under {self.images_dir}")

    def _build_index(self):
        samples = []
        incomplete = 0
        for patient_dir in sorted(self.images_dir.iterdir()):
            if not patient_dir.is_dir():
                continue
            patient_gt = self.gt_dir / patient_dir.name
            for image_path in sorted(patient_dir.glob("*.png")):
                stem = image_path.stem
                mask_paths = [patient_gt / f"{stem}_l{grader}.png" for grader in range(NUM_GRADERS)]
                if not all(path.is_file() for path in mask_paths):
                    incomplete += 1
                    continue
                samples.append((image_path, mask_paths))
        if incomplete:
            print(f"LIDCCrops[{self.split}]: skipped {incomplete} crops without all {NUM_GRADERS} masks")
        return samples

    def _crop_origin(self, height, width, size):
        if not self.train:
            return (height - size) // 2, (width - size) // 2
        top = int(torch.randint(0, height - size + 1, (1,)).item())
        left = int(torch.randint(0, width - size + 1, (1,)).item())
        return top, left

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, mask_paths = self.samples[index]
        image = _load_png(image_path).astype(np.float32) / 255.0
        masks = np.stack([(_load_png(path) > 0).astype(np.float32) for path in mask_paths])
        height, width = image.shape
        if self.crop_size is not None:
            size = self.crop_size
            top, left = self._crop_origin(height, width, size)
            image = image[top:top + size, left:left + size]
            masks = masks[:, top:top + size, left:left + size]
        sample = {
            "image": torch.from_numpy(np.ascontiguousarray(image))[None],
            "masks": torch.from_numpy(np.ascontiguousarray(masks)),
        }
        if self.single_random_grader:
            grader = int(torch.randint(0, NUM_GRADERS, (1,)).item())
            sample["target"] = sample["masks"][grader][None]
            sample["grader"] = grader
        return sample


def binary_entropy(probability, eps=1e-6):
    probability = probability.clamp(eps, 1.0 - eps)
    entropy = -(probability * torch.log(probability) + (1.0 - probability) * torch.log(1.0 - probability))
    return entropy / math.log(2.0)


def compute_disagreement(masks, eps=1e-6):
    if torch.is_tensor(masks):
        probability = masks.float().mean(dim=1, keepdim=True)
        return binary_entropy(probability, eps=eps)
    masks = np.asarray(masks, dtype=np.float32)
    probability = np.clip(masks.mean(axis=1, keepdims=True), eps, 1.0 - eps)
    entropy = -(probability * np.log(probability) + (1.0 - probability) * np.log(1.0 - probability))
    return entropy / np.log(2.0)


def model_uncertainty_from_samples(samples, foreground_channel=1, eps=1e-6, sample_dim=0):
    if sample_dim != 0:
        samples = samples.movedim(sample_dim, 0)
    if samples.shape[2] == 1:
        probabilities = torch.sigmoid(samples)
    else:
        probabilities = F.softmax(samples, dim=2)[:, :, foreground_channel:foreground_channel + 1]
    mean_probability = probabilities.mean(dim=0)
    return binary_entropy(mean_probability, eps=eps)


def disagreement_alignment_loss(model_uncertainty, human_disagreement):
    return F.l1_loss(model_uncertainty, human_disagreement.float())


def assert_shape(test, reference):
    if test.shape != reference.shape:
        raise AssertionError(f"Shape mismatch: {test.shape} and {reference.shape}")


def dice(test, reference, nan_for_nonexisting=False):
    test = np.asarray(test) != 0
    reference = np.asarray(reference) != 0
    assert_shape(test, reference)
    tp = np.logical_and(test, reference).sum()
    fp = np.logical_and(test, ~reference).sum()
    fn = np.logical_and(~test, reference).sum()
    if not test.any() and not reference.any():
        return float("nan") if nan_for_nonexisting else 0.0
    return float(2.0 * tp / (2.0 * tp + fp + fn))


def iou(test, reference, nan_for_nonexisting=False):
    test = np.asarray(test) != 0
    reference = np.asarray(reference) != 0
    assert_shape(test, reference)
    union = np.logical_or(test, reference).sum()
    if union == 0:
        return float("nan") if nan_for_nonexisting else 0.0
    return float(np.logical_and(test, reference).sum() / union)


def binary_iou_distance(test, reference):
    test = np.asarray(test) != 0
    reference = np.asarray(reference) != 0
    union = np.logical_or(test, reference).sum()
    if union == 0:
        return 0.0
    return float(1.0 - np.logical_and(test, reference).sum() / union)


def generalized_energy_distance(samples, references, distance=binary_iou_distance):
    samples = np.asarray(samples)
    references = np.asarray(references)
    sample_reference = np.mean([distance(sample, reference) for sample in samples for reference in references])
    sample_sample = np.mean([distance(a, b) for a in samples for b in samples])
    reference_reference = np.mean([distance(a, b) for a in references for b in references])
    return float(2.0 * sample_reference - sample_sample - reference_reference)


def disagreement_mae(model_uncertainty, human_disagreement):
    return float(np.mean(np.abs(np.asarray(model_uncertainty, dtype=np.float32) - np.asarray(human_disagreement, dtype=np.float32))))


def disagreement_correlation(model_uncertainty, human_disagreement, eps=1e-8):
    a = np.asarray(model_uncertainty, dtype=np.float32).ravel()
    b = np.asarray(human_disagreement, dtype=np.float32).ravel()
    a = a - a.mean()
    b = b - b.mean()
    denom = np.sqrt(np.sum(a ** 2) * np.sum(b ** 2))
    if denom < eps:
        return np.nan
    return float(np.sum(a * b) / denom)


def limit_dataset(dataset, max_items):
    if max_items is None or max_items >= len(dataset):
        return dataset
    return Subset(dataset, range(max_items))


def make_loaders(args):
    train_set = LIDCCrops(split="train", crop_size=args.crop_size, single_random_grader=True)
    eval_set = LIDCCrops(split=args.eval_split, crop_size=args.crop_size, train=False, single_random_grader=True)
    train_set = limit_dataset(train_set, args.max_train)
    eval_set = limit_dataset(eval_set, args.max_val)
    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers, pin_memory=args.device.startswith("cuda"))
    eval_loader = DataLoader(eval_set, batch_size=args.eval_batch_size, shuffle=False, num_workers=args.num_workers, pin_memory=args.device.startswith("cuda"))
    return train_loader, eval_loader


## 5. Model Definition


In [ ]:
def make_onehot(array, labels=None, axis=1, newaxis=False):
    if labels is None:
        labels = np.unique(array)
        labels = list(map(lambda x: x.item(), labels))
    new_shape = list(array.shape)
    if newaxis:
        new_shape.insert(axis, len(labels))
    else:
        new_shape[axis] = new_shape[axis] * len(labels)
    if torch.is_tensor(array):
        new_array = torch.zeros(new_shape, dtype=array.dtype, device=array.device)
    else:
        new_array = np.zeros(new_shape, dtype=array.dtype)
    n_seg_channels = 1 if newaxis else array.shape[axis]
    for seg_channel in range(n_seg_channels):
        for l, label in enumerate(labels):
            new_slc = [slice(None)] * len(new_shape)
            slc = [slice(None)] * len(array.shape)
            new_slc[axis] = seg_channel * len(labels) + l
            if not newaxis:
                slc[axis] = seg_channel
            new_array[tuple(new_slc)] = array[tuple(slc)] == label
    return new_array


def match_to(x, ref, keep_axes=(1,)):
    if isinstance(keep_axes, int):
        keep_axes = (keep_axes,)
    target_shape = list(ref.shape)
    for i in keep_axes:
        target_shape[i] = x.shape[i]
    if x.dim() == 1:
        x = x.unsqueeze(0)
    if x.dim() == 2:
        while x.dim() < len(target_shape):
            x = x.unsqueeze(-1)
    return x.expand(*target_shape).to(device=ref.device, dtype=ref.dtype)


def is_conv(op):
    conv_types = (nn.Conv1d, nn.Conv2d, nn.Conv3d, nn.ConvTranspose1d, nn.ConvTranspose2d, nn.ConvTranspose3d)
    return (type(op) == type and issubclass(op, conv_types)) or type(op) in conv_types


class ConvModule(nn.Module):
    def init_weights(self, init_fn, *args, **kwargs):
        def apply_init(module):
            if is_conv(type(module)):
                module.weight = init_fn(module.weight, *args, **kwargs)
        self.apply(apply_init)


class InjectionConvEncoder(ConvModule):
    def __init__(self, in_channels=1, out_channels=6, depth=4, block_depth=2, num_feature_maps=24,
                 feature_map_multiplier=2, activation_op=nn.LeakyReLU, activation_kwargs=None,
                 norm_op=nn.InstanceNorm2d, norm_kwargs=None, norm_depth=0, conv_op=nn.Conv2d,
                 conv_kwargs=None, pool_op=nn.AvgPool2d, pool_kwargs=None, global_pool_op=nn.AdaptiveAvgPool2d):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        activation_kwargs = {"inplace": True} if activation_kwargs is None else activation_kwargs
        norm_kwargs = {} if norm_kwargs is None else norm_kwargs
        conv_kwargs = {"kernel_size": 3, "padding": 1} if conv_kwargs is None else conv_kwargs
        pool_kwargs = {"kernel_size": 2} if pool_kwargs is None else pool_kwargs
        norm_depth = depth if norm_depth == "full" else norm_depth
        for d in range(depth):
            in_ch = in_channels if d == 0 else num_feature_maps * (feature_map_multiplier ** (d - 1))
            out_ch = num_feature_maps * (feature_map_multiplier ** d)
            layers = []
            if d > 0:
                layers.append(pool_op(**pool_kwargs))
            for b in range(block_depth):
                layers.append(conv_op(in_ch if b == 0 else out_ch, out_ch, **conv_kwargs))
                if norm_op is not None and d < norm_depth:
                    layers.append(norm_op(out_ch, **norm_kwargs))
                layers.append(activation_op(**activation_kwargs))
            if d == depth - 1:
                final_kwargs = dict(conv_kwargs)
                final_kwargs.update({"kernel_size": 1, "padding": 0, "bias": False})
                layers.append(conv_op(out_ch, out_channels, **final_kwargs))
            self.add_module(f"encode_{d}", nn.Sequential(*layers))
        self.add_module("global_pool", global_pool_op(1))

    def forward(self, x):
        for name, module in self._modules.items():
            x = module(x)
        return x


class InjectionUNet(ConvModule):
    def __init__(self, depth=5, in_channels=1, out_channels=2, kernel_size=3, num_feature_maps=24,
                 block_depth=2, num_1x1_at_end=3, injection_channels=6, activation_op=nn.LeakyReLU,
                 activation_kwargs=None, pool_op=nn.AvgPool2d, pool_kwargs=None, norm_op=nn.InstanceNorm2d,
                 norm_kwargs=None, conv_op=nn.Conv2d, conv_kwargs=None, upconv_op=nn.ConvTranspose2d,
                 upconv_kwargs=None, output_activation_op=None, output_activation_kwargs=None):
        super().__init__()
        self.depth = depth
        self.num_feature_maps = num_feature_maps
        self.injection_channels = injection_channels
        self.conv_op = conv_op
        self.last_activations = None
        self.last_features = None
        activation_kwargs = {"inplace": True} if activation_kwargs is None else activation_kwargs
        pool_kwargs = {"kernel_size": 2} if pool_kwargs is None else pool_kwargs
        norm_kwargs = {} if norm_kwargs is None else norm_kwargs
        conv_kwargs = {} if conv_kwargs is None else conv_kwargs
        upconv_kwargs = {} if upconv_kwargs is None else upconv_kwargs
        padding = kernel_size // 2
        for d in range(depth):
            block = []
            if d > 0:
                block.append(pool_op(**pool_kwargs))
            for i in range(block_depth):
                if d == depth - 1 and i > 0:
                    continue
                out_ch = num_feature_maps * 2 ** d
                in_ch = in_channels if d == 0 and i == 0 else (num_feature_maps * 2 ** (d - 1) if i == 0 else out_ch)
                block += [conv_op(in_ch, out_ch, kernel_size, padding=padding, **conv_kwargs)]
                if norm_op is not None:
                    block.append(norm_op(out_ch, **norm_kwargs))
                block.append(activation_op(**activation_kwargs))
            self.add_module(f"encode-{d}", nn.Sequential(*block))
        for d in reversed(range(depth)):
            block = []
            for i in range(block_depth):
                if d == depth - 1 and i > 0:
                    continue
                out_ch = num_feature_maps * 2 ** d
                in_ch = num_feature_maps * 2 ** (d + 1) if i == 0 and d < depth - 1 else out_ch
                block += [conv_op(in_ch, out_ch, kernel_size, padding=padding, **conv_kwargs)]
                if norm_op is not None:
                    block.append(norm_op(out_ch, **norm_kwargs))
                block.append(activation_op(**activation_kwargs))
            if d > 0:
                block.append(upconv_op(out_ch, out_ch // 2, kernel_size, 2, padding=padding, output_padding=1, **upconv_kwargs))
            self.add_module(f"decode-{d}", nn.Sequential(*block))
        in_ch = num_feature_maps + injection_channels
        for i in range(num_1x1_at_end):
            out_ch = out_channels if i == num_1x1_at_end - 1 else num_feature_maps
            self.add_module(f"reduce-{i}", conv_op(in_ch, out_ch, 1, bias=True))
            if i != num_1x1_at_end - 1:
                self.add_module(f"reduce-{i}-nonlin", activation_op(**activation_kwargs))
            in_ch = out_ch
        if output_activation_op is not None:
            self.add_module("output-activation", output_activation_op(**(output_activation_kwargs or {})))

    def reset(self):
        self.last_activations = None
        self.last_features = None

    def forward(self, x, injection=None, reuse_last_activations=False, store_activations=False):
        if self.last_activations is None or not reuse_last_activations:
            enc = [x]
            for i in range(self.depth - 1):
                enc.append(self._modules[f"encode-{i}"](enc[-1]))
            bottom = self._modules[f"encode-{self.depth - 1}"](enc[-1])
            x = self._modules[f"decode-{self.depth - 1}"](bottom)
            for i in reversed(range(self.depth - 1)):
                x = self._modules[f"decode-{i}"](torch.cat((enc[-(self.depth - 1 - i)], x), 1))
            self.last_features = x
            if store_activations:
                self.last_activations = x.detach()
        else:
            x = self.last_activations
        if self.injection_channels > 0:
            x = torch.cat((x, match_to(injection, x, (0, 1))), 1)
        for i in range(3):
            x = self._modules[f"reduce-{i}"](x)
            if f"reduce-{i}-nonlin" in self._modules:
                x = self._modules[f"reduce-{i}-nonlin"](x)
        if "output-activation" in self._modules:
            x = self._modules["output-activation"](x)
        return x


class ProbabilisticSegmentationNet(ConvModule):
    def __init__(self, in_channels=1, out_channels=2, num_feature_maps=32, latent_size=6, depth=5):
        super().__init__()
        self.latent_distribution = distributions.Normal
        self.task_net = InjectionUNet(
            in_channels=in_channels, out_channels=out_channels, num_feature_maps=num_feature_maps,
            injection_channels=latent_size, depth=depth, output_activation_op=nn.LogSoftmax,
            output_activation_kwargs={"dim": 1}
        )
        self.prior_net = InjectionConvEncoder(
            in_channels=in_channels, out_channels=2 * latent_size, depth=depth,
            num_feature_maps=num_feature_maps, norm_depth=0
        )
        self.posterior_net = InjectionConvEncoder(
            in_channels=in_channels + out_channels, out_channels=2 * latent_size, depth=depth,
            num_feature_maps=num_feature_maps, norm_depth=0
        )
        self._prior = None
        self._posterior = None

    @property
    def prior(self):
        return self._prior

    @property
    def posterior(self):
        return self._posterior

    @property
    def last_activations(self):
        return self.task_net.last_activations

    def train(self, mode=True):
        super().train(mode)
        self.reset()
        return self

    def reset(self):
        self.task_net.reset()
        self._prior = None
        self._posterior = None

    def encode_prior(self, input_):
        rep = self.prior_net(input_)
        mean, logvar = torch.split(rep, rep.shape[1] // 2, dim=1)
        self._prior = self.latent_distribution(mean, logvar.mul(0.5).exp())
        return self._prior

    def encode_posterior(self, input_, seg, make_onehot_classes=(0, 1)):
        seg = make_onehot(seg, make_onehot_classes).float()
        rep = self.posterior_net(torch.cat((input_, seg), 1))
        mean, logvar = torch.split(rep, rep.shape[1] // 2, dim=1)
        self._posterior = self.latent_distribution(mean, logvar.mul(0.5).exp())
        return self._posterior

    def forward(self, input_, seg=None, make_onehot=True, make_onehot_classes=(0, 1)):
        self.encode_prior(input_)
        if self.training:
            self.encode_posterior(input_, seg, make_onehot_classes=make_onehot_classes)
            sample = self.posterior.rsample()
        else:
            sample = self.prior.loc
        return self.task_net(input_, sample, store_activations=not self.training)

    def sample_prior(self, n_samples=1, input_=None, out_device=None):
        if out_device is None:
            out_device = input_.device
        with torch.no_grad():
            if self.prior is None or input_ is not None:
                self.encode_prior(input_)
            outputs = []
            outputs.append(self.task_net(input_, self.prior.sample(), reuse_last_activations=False, store_activations=True).to(out_device))
            while len(outputs) < n_samples:
                outputs.append(self.task_net(input_, self.prior.sample(), reuse_last_activations=True).to(out_device))
            return outputs[0] if n_samples == 1 else outputs


class DisagreementAwareProbabilisticSegmentationNet(ProbabilisticSegmentationNet):
    def __init__(self, *args, disagreement_channels=1, foreground_channel=1, **kwargs):
        self.disagreement_channels = disagreement_channels
        self.foreground_channel = foreground_channel
        super().__init__(*args, **kwargs)
        self.disagreement_head = nn.Conv2d(self.task_net.num_feature_maps, disagreement_channels, kernel_size=1)

    def predict_disagreement(self):
        if self.task_net.last_features is None:
            raise ValueError("Run a forward pass before predicting disagreement.")
        return torch.sigmoid(self.disagreement_head(self.task_net.last_features))

    def sample_prior_train(self, input_, n_samples=4):
        if self.prior is None:
            self.encode_prior(input_)
        return torch.stack([self.task_net(input_, self.prior.rsample(), reuse_last_activations=False) for _ in range(n_samples)], dim=0)

    def disagreement_losses(self, input_, masks, n_samples=4, lambda_disagreement=1.0, lambda_alignment=0.0):
        human_disagreement = compute_disagreement(masks)
        predicted_disagreement = self.predict_disagreement()
        loss_disagreement = F.mse_loss(predicted_disagreement, human_disagreement.float())
        loss_alignment = predicted_disagreement.new_tensor(0.0)
        model_uncertainty = None
        if lambda_alignment != 0.0:
            samples = self.sample_prior_train(input_, n_samples=n_samples)
            model_uncertainty = model_uncertainty_from_samples(samples, foreground_channel=self.foreground_channel)
            loss_alignment = disagreement_alignment_loss(model_uncertainty, human_disagreement)
        loss = lambda_disagreement * loss_disagreement + lambda_alignment * loss_alignment
        metrics = {"loss_disagreement": loss_disagreement.detach(), "loss_alignment": loss_alignment.detach()}
        if model_uncertainty is not None:
            metrics["model_uncertainty"] = model_uncertainty.detach()
        return loss, metrics


def make_model(args):
    model_cls = ProbabilisticSegmentationNet if args.variant == "baseline" else DisagreementAwareProbabilisticSegmentationNet
    return model_cls(
        in_channels=1,
        out_channels=2,
        num_feature_maps=args.feature_maps,
        latent_size=args.latent_size,
        depth=args.depth,
    ).to(args.device)


## 6. Training, Checkpointing, Evaluation, And Figures


In [ ]:
VARIANTS = ("baseline", "head", "full")
VARIANT_SETTINGS = {
    "baseline": {},
    "head": {"lambda_disagreement": 0.5},
    "full": {"lambda_disagreement": 0.5, "lambda_alignment": 0.5},
}


def batch_to_device(batch, device):
    return {key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value for key, value in batch.items()}


def kl_loss(model):
    kl = distributions.kl_divergence(model.posterior, model.prior)
    return kl.reshape(kl.shape[0], -1).sum(dim=1).mean()


def set_lr(optimizer, step, args):
    if args.lr_final >= args.lr:
        return args.lr
    decay_every = max(1, args.steps // args.lr_decay_steps)
    decay_index = min((step - 1) // decay_every, args.lr_decay_steps)
    gamma = (args.lr_final / args.lr) ** (1.0 / args.lr_decay_steps)
    lr = args.lr * (gamma ** decay_index)
    for group in optimizer.param_groups:
        group["lr"] = lr
    return lr


def train_step(model, batch, optimizer, criterion, args, step):
    model.train()
    model.reset()
    optimizer.zero_grad()
    lr = set_lr(optimizer, step, args)
    image = batch["image"]
    target = batch["target"].long()
    masks = batch["masks"]
    prediction = model(image, target, make_onehot_classes=(0, 1))
    loss_seg = criterion(prediction, target[:, 0].long())
    loss_kl = kl_loss(model)
    loss = loss_seg + args.beta * loss_kl
    log = {
        "step": step,
        "lr": lr,
        "loss": loss.detach(),
        "loss_seg": loss_seg.detach(),
        "loss_kl": loss_kl.detach(),
        "loss_disagreement": prediction.new_tensor(0.0),
        "loss_alignment": prediction.new_tensor(0.0),
    }
    if args.variant in ("head", "full"):
        lambda_alignment = args.lambda_alignment if args.variant == "full" else 0.0
        loss_aux, aux_metrics = model.disagreement_losses(
            image, masks, n_samples=args.train_samples,
            lambda_disagreement=args.lambda_disagreement,
            lambda_alignment=lambda_alignment,
        )
        loss = loss + loss_aux
        log["loss"] = loss.detach()
        log["loss_disagreement"] = aux_metrics["loss_disagreement"]
        log["loss_alignment"] = aux_metrics["loss_alignment"]
    loss.backward()
    optimizer.step()
    return {key: float(value.detach().cpu()) if torch.is_tensor(value) else value for key, value in log.items()}


def sample_masks(model, image, n_samples):
    outputs = model.sample_prior(n_samples, input_=image, out_device=image.device)
    outputs = torch.stack(outputs, dim=0)
    masks = torch.argmax(outputs, dim=2)
    return outputs, masks


def evaluate(model, loader, args):
    model.eval()
    dice_scores, iou_scores, ged_scores = [], [], []
    uncertainty_maes, uncertainty_corrs = [], []
    predicted_maes, predicted_corrs = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="eval", leave=False, dynamic_ncols=True):
            batch = batch_to_device(batch, args.device)
            image = batch["image"]
            masks = batch["masks"]
            model.reset()
            prediction = model(image)
            pred_mask = torch.argmax(prediction, dim=1).cpu().numpy()
            grader_masks = masks.cpu().numpy().astype(bool)
            outputs, prior_masks = sample_masks(model, image, args.eval_samples)
            model_uncertainty = model_uncertainty_from_samples(outputs).cpu().numpy()
            human_disagreement = compute_disagreement(masks).cpu().numpy()
            predicted_disagreement = None
            if args.variant in ("head", "full"):
                predicted_disagreement = model.predict_disagreement().cpu().numpy()
            prior_masks = prior_masks.cpu().numpy().astype(bool)
            for i in range(image.shape[0]):
                for grader in range(grader_masks.shape[1]):
                    dice_scores.append(dice(pred_mask[i] != 0, grader_masks[i, grader]))
                    iou_scores.append(iou(pred_mask[i] != 0, grader_masks[i, grader]))
                ged_scores.append(generalized_energy_distance(prior_masks[:, i], grader_masks[i]))
                uncertainty_maes.append(disagreement_mae(model_uncertainty[i, 0], human_disagreement[i, 0]))
                uncertainty_corrs.append(disagreement_correlation(model_uncertainty[i, 0], human_disagreement[i, 0]))
                if predicted_disagreement is not None:
                    predicted_maes.append(disagreement_mae(predicted_disagreement[i, 0], human_disagreement[i, 0]))
                    predicted_corrs.append(disagreement_correlation(predicted_disagreement[i, 0], human_disagreement[i, 0]))
    result = {
        "val_dice": float(np.nanmean(dice_scores)),
        "val_iou": float(np.nanmean(iou_scores)),
        "val_ged": float(np.nanmean(ged_scores)),
        "val_uncertainty_mae": float(np.nanmean(uncertainty_maes)),
        "val_uncertainty_corr": float(np.nanmean(uncertainty_corrs)),
    }
    if predicted_maes:
        result["val_predicted_disagreement_mae"] = float(np.nanmean(predicted_maes))
        result["val_predicted_disagreement_corr"] = float(np.nanmean(predicted_corrs))
    return result


def append_csv(path, row):
    exists = Path(path).exists()
    with open(path, "a", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=sorted(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def save_checkpoint(path, model, optimizer, step, args, metrics):
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,
        "args": vars(args),
        "metrics": metrics,
        "rng_state": {
            "python": random.getstate(),
            "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        },
    }, path)


def torch_load(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def load_checkpoint(path, model, optimizer=None, device="cpu", load_optimizer=True):
    checkpoint = torch_load(path, device)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and load_optimizer and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    rng_state = checkpoint.get("rng_state")
    if rng_state is not None:
        random.setstate(rng_state["python"])
        np.random.set_state(rng_state["numpy"])
        torch.set_rng_state(rng_state["torch"])
        if rng_state.get("cuda") is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(rng_state["cuda"])
    return checkpoint


def best_ged_from_history(path):
    path = Path(path)
    if not path.exists():
        return None
    values = []
    with open(path, newline="") as handle:
        for row in csv.DictReader(handle):
            if row.get("val_ged") not in (None, ""):
                values.append(float(row["val_ged"]))
    return min(values) if values else None


def make_args(variant, eval_split="val"):
    settings = VARIANT_SETTINGS[variant]
    return SimpleNamespace(
        variant=variant,
        data_root=str(DATA_ROOT),
        out_dir=str(OUTPUT_ROOT / "lidc_ablation" / variant),
        eval_split=eval_split,
        steps=CFG["steps"],
        batch_size=CFG["batch_size"],
        eval_batch_size=CFG["eval_batch_size"],
        crop_size=128,
        max_train=CFG["max_train"],
        max_val=CFG["max_val"],
        num_workers=0,
        device=DEVICE,
        feature_maps=CFG["feature_maps"],
        latent_size=CFG["latent_size"],
        depth=CFG["depth"],
        lr=1e-4,
        lr_final=1e-6,
        lr_decay_steps=5,
        weight_decay=1e-5,
        beta=1.0,
        lambda_disagreement=settings.get("lambda_disagreement", 0.5),
        lambda_alignment=settings.get("lambda_alignment", 0.5),
        train_samples=CFG["train_samples"],
        eval_samples=CFG["eval_samples"],
        eval_every=CFG["eval_every"],
        save_every=CFG["save_every"],
        seed=SEED,
    )


def train_variant(variant):
    args = make_args(variant, eval_split="val")
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "args.json", "w") as handle:
        json.dump(vars(args), handle, indent=2, sort_keys=True)
    train_loader, val_loader = make_loaders(args)
    model = make_model(args)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    criterion = nn.NLLLoss()
    latest = out_dir / "latest_checkpoint.pt"
    history_path = out_dir / "history.csv"
    best_ged = best_ged_from_history(history_path)
    start_step = 1
    if latest.exists() and not FORCE_RETRAIN:
        checkpoint = load_checkpoint(latest, model, optimizer, device=args.device)
        start_step = int(checkpoint.get("step", 0)) + 1
        if best_ged is None:
            best_ged = (checkpoint.get("metrics") or {}).get("val_ged")
        print(f"{variant}: resumed from step {start_step - 1}")
    if start_step > args.steps:
        print(f"{variant}: already complete at step {start_step - 1}")
        return
    print(f"{variant}: {len(train_loader.dataset)} train / {len(val_loader.dataset)} val samples")
    train_iter = iter(train_loader)
    started = time.time()
    progress = tqdm(range(start_step, args.steps + 1), desc=f"train {variant}", initial=start_step - 1, total=args.steps, dynamic_ncols=True)
    for step in progress:
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)
        batch = batch_to_device(batch, args.device)
        row = train_step(model, batch, optimizer, criterion, args, step)
        progress.set_postfix(loss=f"{row['loss']:.4f}", lr=f"{row['lr']:.2e}")
        should_eval = step == 1 or step % args.eval_every == 0 or step == args.steps
        if should_eval:
            metrics = evaluate(model, val_loader, args)
            row.update(metrics)
            row["elapsed_min"] = (time.time() - started) / 60.0
            append_csv(history_path, row)
            tqdm.write("{variant} step={step} loss={loss:.4f} dice={val_dice:.4f} iou={val_iou:.4f} ged={val_ged:.4f} unc_mae={val_uncertainty_mae:.4f}".format(variant=variant, **row))
            if best_ged is None or metrics["val_ged"] < best_ged:
                best_ged = metrics["val_ged"]
                save_checkpoint(out_dir / "best_checkpoint.pt", model, optimizer, step, args, metrics)
        if step % args.save_every == 0 or step == args.steps:
            save_checkpoint(latest, model, optimizer, step, args, row)


## 7. Visual Sanity Check For Human Disagreement


In [ ]:
preview_dir = OUTPUT_ROOT / "preview"
preview_dir.mkdir(parents=True, exist_ok=True)
ds_preview = LIDCCrops(split="val", train=False, single_random_grader=False)
sample = ds_preview[0]
d_map = compute_disagreement(sample["masks"][None]).numpy()[0, 0]
fig, axes = plt.subplots(1, 6, figsize=(14, 2.4))
panels = [sample["image"][0].numpy()] + [sample["masks"][i].numpy() for i in range(4)] + [d_map]
titles = ["image", "mask 1", "mask 2", "mask 3", "mask 4", "human D"]
for ax, panel, title in zip(axes, panels, titles):
    cmap = "magma" if title == "human D" else "gray"
    ax.imshow(panel, cmap=cmap, vmin=0 if cmap == "magma" else None, vmax=1 if cmap == "magma" else None)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
preview_path = preview_dir / "disagreement_preview.png"
fig.savefig(preview_path, dpi=160)
plt.close(fig)
display(DisplayImage(filename=str(preview_path)))


## 8. Train The Three Ablations


In [ ]:
for variant in VARIANTS:
    train_variant(variant)


## 9. Test Evaluation Table


In [ ]:
def best_or_latest(variant):
    out_dir = OUTPUT_ROOT / "lidc_ablation" / variant
    best = out_dir / "best_checkpoint.pt"
    latest = out_dir / "latest_checkpoint.pt"
    return best if best.exists() else latest


def evaluate_checkpoint(variant, split=EVAL_SPLIT):
    ckpt_path = best_or_latest(variant)
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Missing checkpoint for {variant}: {ckpt_path}")
    checkpoint = torch_load(ckpt_path, DEVICE)
    saved_args = dict(checkpoint.get("args") or {})
    saved_args.update({
        "variant": variant,
        "device": DEVICE,
        "eval_split": split,
        "eval_samples": CFG["eval_samples"],
        "eval_batch_size": CFG["eval_batch_size"],
        "max_val": CFG["max_val"],
        "num_workers": 0,
    })
    args = SimpleNamespace(**saved_args)
    _, loader = make_loaders(args)
    model = make_model(args)
    model.load_state_dict(checkpoint["model_state_dict"])
    metrics = evaluate(model, loader, args)
    return {"variant": variant, "checkpoint": str(ckpt_path), **metrics}

results = [evaluate_checkpoint(variant) for variant in VARIANTS]
for row in results:
    print(row)

metrics_path = OUTPUT_ROOT / "lidc_ablation" / f"{EVAL_SPLIT}_metrics_{RUN_MODE}.csv"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
fieldnames = sorted({key for row in results for key in row})
with open(metrics_path, "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)
print("Wrote", metrics_path)


## 10. Qualitative Figure Generation


In [ ]:
def load_model_for_figures(checkpoint_path, variant):
    checkpoint = torch_load(checkpoint_path, DEVICE)
    saved_args = dict(checkpoint.get("args") or {})
    saved_args.update({"variant": variant, "device": DEVICE})
    args = SimpleNamespace(**saved_args)
    model = make_model(args)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint


def sample_model_for_figure(model, image, n_samples, has_disagreement_head):
    with torch.no_grad():
        model.reset()
        outputs, masks = sample_masks(model, image, n_samples)
        uncertainty = model_uncertainty_from_samples(outputs).cpu().numpy()[0, 0]
        predicted_disagreement = None
        if has_disagreement_head:
            predicted_disagreement = model.predict_disagreement().cpu().numpy()[0, 0]
    return masks[:, 0].cpu().numpy().astype(np.float32), uncertainty, predicted_disagreement


def disagreement_score(dataset, index):
    sample = dataset[index]
    return float(compute_disagreement(sample["masks"][None]).numpy()[0, 0].mean())


def choose_high_disagreement_indices(dataset, count):
    scores = []
    for index in tqdm(range(len(dataset)), desc="rank disagreement", dynamic_ncols=True):
        scores.append((disagreement_score(dataset, index), index))
    scores.sort(reverse=True)
    return [index for _, index in scores[:count]]


def add_panel(ax, image, title, cmap="gray", vmin=None, vmax=None):
    ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=8)
    ax.axis("off")


def make_case_figure(out_path, image, masks, human_disagreement, baseline_samples, baseline_uncertainty,
                     full_samples, full_uncertainty, predicted_disagreement):
    n_samples = baseline_samples.shape[0]
    ncols = max(6, n_samples + 2)
    fig, axes = plt.subplots(4, ncols, figsize=(2.1 * ncols, 8.2))
    for ax in axes.ravel():
        ax.axis("off")
    add_panel(axes[0, 0], image, "input")
    for grader in range(masks.shape[0]):
        add_panel(axes[0, grader + 1], masks[grader], f"human M{grader + 1}")
    add_panel(axes[0, 5], human_disagreement, "human D", cmap="magma", vmin=0.0, vmax=1.0)
    axes[1, 0].set_title("baseline samples", fontsize=8)
    for i in range(n_samples):
        add_panel(axes[1, i + 1], baseline_samples[i], f"S{i + 1}")
    add_panel(axes[1, n_samples + 1], baseline_uncertainty, "baseline U", cmap="magma", vmin=0.0, vmax=1.0)
    axes[2, 0].set_title("full samples", fontsize=8)
    for i in range(n_samples):
        add_panel(axes[2, i + 1], full_samples[i], f"S{i + 1}")
    add_panel(axes[2, n_samples + 1], full_uncertainty, "full U", cmap="magma", vmin=0.0, vmax=1.0)
    add_panel(axes[3, 0], predicted_disagreement, "predicted D", cmap="magma", vmin=0.0, vmax=1.0)
    add_panel(axes[3, 1], np.abs(full_uncertainty - human_disagreement), "|U-D|", cmap="magma")
    add_panel(axes[3, 2], np.abs(predicted_disagreement - human_disagreement), "|Dhat-D|", cmap="magma")
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def generate_figures():
    fig_dir = OUTPUT_ROOT / "lidc_figures" / RUN_MODE
    fig_dir.mkdir(parents=True, exist_ok=True)
    baseline, baseline_ckpt = load_model_for_figures(best_or_latest("baseline"), "baseline")
    full, full_ckpt = load_model_for_figures(best_or_latest("full"), "full")
    crop_size = (baseline_ckpt.get("args") or {}).get("crop_size", 128)
    dataset = LIDCCrops(split=EVAL_SPLIT, crop_size=crop_size, train=False, single_random_grader=False)
    indices = choose_high_disagreement_indices(dataset, CFG["figure_cases"])
    for index in tqdm(indices, desc="figures", dynamic_ncols=True):
        sample = dataset[index]
        image = sample["image"][None].to(DEVICE)
        masks = sample["masks"].numpy()
        human_disagreement = compute_disagreement(sample["masks"][None]).numpy()[0, 0]
        baseline_samples, baseline_uncertainty, _ = sample_model_for_figure(baseline, image, CFG["figure_samples"], False)
        full_samples, full_uncertainty, predicted_disagreement = sample_model_for_figure(full, image, CFG["figure_samples"], True)
        out_path = fig_dir / f"case_{index:04d}.png"
        make_case_figure(out_path, sample["image"][0].numpy(), masks, human_disagreement,
                         baseline_samples, baseline_uncertainty, full_samples, full_uncertainty,
                         predicted_disagreement)
        print("wrote", out_path)
    return fig_dir

fig_dir = generate_figures()


## 11. Display Generated Figures


In [ ]:
figure_paths = sorted(fig_dir.glob("case_*.png"))
print(f"Generated {len(figure_paths)} figures in {fig_dir}")
for path in figure_paths[: min(4, len(figure_paths))]:
    print(path)
    display(DisplayImage(filename=str(path)))


## 12. Outputs

The notebook writes:

- `outputs/lidc_ablation/<variant>/history.csv`
- `outputs/lidc_ablation/<variant>/best_checkpoint.pt`
- `outputs/lidc_ablation/<variant>/latest_checkpoint.pt`
- `outputs/lidc_ablation/<split>_metrics_<mode>.csv`
- `outputs/lidc_figures/<mode>/case_*.png`

For final submission results, set `RUN_MODE = "final"` and run the notebook on a GPU runtime.
